# 🛠️ Preprocessing Lengkap — Dataset IndoToxic2024
**Proyek:** Indonesian Hate Speech Analyzer (Kelompok 6) | **Model target:** XLM-RoBERTa

Notebook ini mengeksekusi preprocessing **secara berurutan (beriringan)** — setiap langkah ditampilkan:
1. **Kode transformasi**
2. **Tabel BEFORE vs AFTER** pada sampel teks yang sama, supaya efeknya langsung terlihat.

Di bagian akhir ada **eksperimen khusus**: menguji apakah **tidak melakukan lowercasing** berpengaruh terhadap tokenisasi **XLM-RoBERTa** — karena arsitektur ini punya karakteristik tokenizer yang berbeda dari BERT-uncased.

> Urutan mengikuti hasil EDA sebelumnya: **dedup & conflicting label dulu (sebelum split)**, baru normalisasi teks, baru split train/test.

## Bagian 0 — Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import re
import ast
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

print("✅ Library dasar dimuat.")

✅ Library dasar dimuat.


In [2]:
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
file_path = project_root / 'data' / 'raw' / 'indotoxic2024_annotated_data_v2_final.csv'
local_fallback = current_dir / 'indotoxic2024_annotated_data_v2_final.csv'

if file_path.exists():
    data_source = file_path
elif local_fallback.exists():
    data_source = local_fallback
else:
    raise FileNotFoundError(f"Dataset tidak ditemukan di {file_path} maupun {local_fallback}.")

df = pd.read_csv(data_source)
print(f"Sumber   : {data_source.resolve()}")
print(f"Baris    : {df.shape[0]:,} | Kolom: {df.shape[1]}")
df.head(3)

Sumber   : C:\Users\HP\projects-SistemCerdas\hate-speech-analyzer\data\raw\indotoxic2024_annotated_data_v2_final.csv
Baris    : 28,448 | Kolom: 14


,text_id,annotators_id,text,initial_paragraph,topic,is_noise_or_spam_text,related_to_election_2024,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,“️Jurnalis Hana Mahamed kembali tampil di televisi dengan wajah terluka akibat serangan Israel di Gaza Keinginan yan...,UNKNOWN,"['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']"
1,1-10,"['7', '15']","Elektabilitas Paslon 02 sebentar lagi terjun bebas gara2 Gemoy, joget, susu gratis, makan gratis, politik dinasti, c...",NaN,Disabilitas,"['0', '0']","['1', '1']","['0', '0']","['1', '1']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']"
2,1-100,"['7', '15']",Gini aja deh @KPU_ID usul aja nih. Gimana kalo misalnya gak usah ada debat aja kalo ga boleh dikritik ? Diubah jadi ...,NaN,Disabilitas,"['0', '0']","['1', '0']","['1', '0']","['0', '0']","['0', '0']","['0', '0']","['1', '0']","['0', '0']","['0', '0']"


In [3]:
def parse_annotation_list(s):
    try:
        return [int(x) for x in ast.literal_eval(s)]
    except (ValueError, SyntaxError):
        return []

def majority_vote(s):
    votes = parse_annotation_list(s)
    if not votes:
        return np.nan
    avg = np.mean(votes)
    return 1 if avg > 0.5 else (0 if avg < 0.5 else 0.5)

label_columns = ['toxicity','identity_attack','threat_incitement_to_violence','insults',
                  'profanity_obscenity','sexually_explicit','polarized',
                  'related_to_election_2024','is_noise_or_spam_text']
for col in label_columns:
    df[f'{col}_label'] = df[col].apply(majority_vote)

text_col = 'text'
main_label_col = 'toxicity_label'
print("Label utama:", main_label_col)
print(f"Baris awal sebelum preprocessing: {len(df):,}")

Label utama: toxicity_label
Baris awal sebelum preprocessing: 28,448


## STEP 1 — Buang Teks Tidak Valid (NaN/Kosong)
Sesuai temuan EDA: hanya sebagian kecil baris yang bermasalah, dampaknya minimal tapi tetap harus dibersihkan agar tidak error saat tokenisasi.

In [4]:
before_n = len(df)

text_as_str = df[text_col].fillna('').astype(str)
is_blank = text_as_str.eq('')
is_whitespace = text_as_str.str.fullmatch(r'\s+', na=False)
is_symbol_only = text_as_str.str.fullmatch(r'[^\w\s]+', na=False)
invalid_mask = df[text_col].isna() | is_blank | is_whitespace | is_symbol_only

df = df[~invalid_mask].reset_index(drop=True)

print(f"BEFORE : {before_n:,} baris")
print(f"AFTER  : {len(df):,} baris  (dibuang: {before_n - len(df)} baris teks tidak valid)")

BEFORE : 28,448 baris
AFTER  : 28,447 baris  (dibuang: 1 baris teks tidak valid)


## STEP 2 — Deduplikasi & Resolusi Conflicting Label (WAJIB sebelum split)
Kita normalisasi teks dulu (lowercase + hapus URL/mention/simbol) untuk mendeteksi *near-duplicate*, lalu:
1. Kelompokkan baris berdasarkan teks ternormalisasi.
2. Untuk grup dengan **conflicting label** (label target berbeda-beda dalam satu grup teks yang sama), gunakan **modus (label paling sering muncul)** sebagai label konsensus akhir — bukan dihapus asal.
3. Ambil satu representasi (baris pertama) per grup untuk mengurangi duplikasi & risiko data leakage.

In [5]:
def normalize_for_dup(x):
    x = str(x).lower()
    x = re.sub(r'https?://\S+|www\.\S+', ' URL ', x)
    x = re.sub(r'@\w+', ' USER ', x)
    x = re.sub(r'\s+', ' ', x)
    x = re.sub(r'[^\w\s]', '', x)
    return x.strip()

before_n = len(df)
df['_norm_text'] = df[text_col].map(normalize_for_dup)

# --- Resolusi conflicting label per grup teks ternormalisasi ---
def resolve_label(series):
    modes = series.mode()
    return modes.iloc[0] if len(modes) else np.nan

group_label = df.groupby('_norm_text')[main_label_col].transform(resolve_label)
n_conflict_rows = int((df[main_label_col] != group_label).sum())
df[main_label_col] = group_label

# --- Ambil 1 representasi per grup (baris pertama) ---
df_dedup = df.drop_duplicates(subset='_norm_text', keep='first').reset_index(drop=True)

print(f"BEFORE dedup : {before_n:,} baris")
print(f"AFTER dedup  : {len(df_dedup):,} baris  (dibuang: {before_n - len(df_dedup):,} duplikat/near-duplicate)")
print(f"Baris yang label-nya disesuaikan lewat resolusi konflik (modus): {n_conflict_rows:,}")

df = df_dedup.drop(columns='_norm_text')

BEFORE dedup : 28,447 baris
AFTER dedup  : 25,998 baris  (dibuang: 2,449 duplikat/near-duplicate)
Baris yang label-nya disesuaikan lewat resolusi konflik (modus): 427


## STEP 3 — Fungsi Pembersihan Teks Dasar (URL, Mention, Hashtag, Newline, Whitespace)
Dibuat sebagai **satu fungsi bertahap** agar bisa dibandingkan before/after per bagian. URL & mention diganti token khusus (bukan dihapus polos) agar model masih tahu "di sini ada tautan/sebutan pengguna" tanpa membawa noise string mentah.

In [6]:
def clean_url_mention_hashtag(text):
    t = str(text)
    t = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', t)
    t = re.sub(r'@\w+', ' <USER> ', t)
    t = re.sub(r'#(\w+)', r'\1', t)          # hashtag: buang simbol '#', pertahankan kata
    t = re.sub(r'[\n\r]+', ' ', t)            # newline -> spasi
    t = re.sub(r'\s+', ' ', t).strip()         # rapikan whitespace berlebih
    return t

sample_idx = df.sample(5, random_state=42).index
before_after = pd.DataFrame({
    'BEFORE': df.loc[sample_idx, text_col],
    'AFTER (url/mention/hashtag)': df.loc[sample_idx, text_col].map(clean_url_mention_hashtag)
})
display(before_after)

,BEFORE,AFTER (url/mention/hashtag)
4148,Bismillah 👳 CARA PERSATUKAN UMMAT HANYA DENGAN DAKWAH !! Kenapa mereka ? #YAHUDI #SYIAH #KOMUNIS Hari ini semakin be...,Bismillah 👳 CARA PERSATUKAN UMMAT HANYA DENGAN DAKWAH !! Kenapa mereka ? YAHUDI SYIAH KOMUNIS Hari ini semakin beran...
7839,"Tanggal 28 kok masuk kerja, bossmu Yahudi taa?","Tanggal 28 kok masuk kerja, bossmu Yahudi taa?"
18896,"Indonesia Mengutuk Serangan Israel ke Gaza, Siap Kirim Bantuan ke Palestina","Indonesia Mengutuk Serangan Israel ke Gaza, Siap Kirim Bantuan ke Palestina"
4194,"When sainganmu cewe yang gila kerja, bisa cari uang sendiri, me time and self reward tiap minggu royal, bisa ngurus ...","When sainganmu cewe yang gila kerja, bisa cari uang sendiri, me time and self reward tiap minggu royal, bisa ngurus ..."
8407,"Prediksi 🇨🇳 China Minggu, 15, Oct, 2023 Selalu Utamakan Prediksi Sendiri Bosku !! Semoga JP Paus !! Link Situs Predi...","Prediksi 🇨🇳 China Minggu, 15, Oct, 2023 Selalu Utamakan Prediksi Sendiri Bosku !! Semoga JP Paus !! Link Situs Predi..."


## STEP 4 — Normalisasi Tanda Baca Berlebihan
Bukan dihapus total — tanda baca berulang (`!!!!`, `???`) **dibatasi maksimal 2 pengulangan** supaya sinyal intensitas/emosi tetap ada, tapi variasi liar (`!!!!!!!!`) tidak membuat vocabulary meledak.

In [7]:
def normalize_punctuation(text):
    return re.sub(r'([!?.,])\1{2,}', r'\1\1', str(text))

sample_punct = df[df[text_col].str.contains(r'([!?.,])\1{2,}', regex=True, na=False)].sample(
    min(5, (df[text_col].str.contains(r'([!?.,])\1{2,}', regex=True, na=False)).sum()), random_state=1
)
display(pd.DataFrame({
    'BEFORE': sample_punct[text_col],
    'AFTER (punctuation)': sample_punct[text_col].map(normalize_punctuation)
}))

,BEFORE,AFTER (punctuation)
9661,Semoga doa kita semua bisa membebaskan dan selamatkan Palestine ... Dan semoga Zionis Israel hancur ://,Semoga doa kita semua bisa membebaskan dan selamatkan Palestine .. Dan semoga Zionis Israel hancur ://
6693,Jokowi tahun 2014 tahun 2019 ngelarang kita memilih Prabowo pada saat Jokowi mencalonkan diri sebagai presiden karen...,Jokowi tahun 2014 tahun 2019 ngelarang kita memilih Prabowo pada saat Jokowi mencalonkan diri sebagai presiden karen...
23479,@9191_dwi @VIVAcoid .....??????? Kaum rohingya bisa hidup di mana saja asal ada tanah buat mereka tinggal. Buang mer...,@9191_dwi @VIVAcoid ..?? Kaum rohingya bisa hidup di mana saja asal ada tanah buat mereka tinggal. Buang mereka ke s...
1498,"@doniriw MENAKLUKKAN NEGERI ZIONIS ITU MUDAH SAJA, TAPI... MENAKLUKKAN NEGERI 2IONIS ITU MUDAH SAJA, TAPI\n\n© Doni ...","@doniriw MENAKLUKKAN NEGERI ZIONIS ITU MUDAH SAJA, TAPI.. MENAKLUKKAN NEGERI 2IONIS ITU MUDAH SAJA, TAPI\n\n© Doni R..."
3357,Katolik semakin dipersulit di Israel Haruskah kita membela Israel? Gereja Katolik dan gereja kristen lainnya mengala...,Katolik semakin dipersulit di Israel Haruskah kita membela Israel? Gereja Katolik dan gereja kristen lainnya mengala...


## STEP 5 — Konversi Emoji ke Token Deskriptif
Emoji **tidak dihapus** karena terbukti di EDA (~20% dokumen) berpotensi membawa sinyal emosi. Kita coba pakai library `emoji` (jika tersedia) untuk mengonversi emoji menjadi deskripsi teks (mis. 😡 → `:pengsi_marah:`). Jika library tidak tersedia, emoji dibiarkan apa adanya (byte-fallback tokenizer XLM-R tetap bisa memprosesnya).

In [8]:
try:
    import emoji
    def convert_emoji(text):
        return emoji.demojize(str(text), language='id') if hasattr(emoji, 'demojize') else emoji.demojize(str(text))
    print("✅ Library 'emoji' tersedia — emoji akan dikonversi ke token deskriptif.")
except ImportError:
    print("⚠️ Library 'emoji' belum terinstal. Jalankan: pip install emoji")
    print("   Emoji akan DIBIARKAN apa adanya (tidak fatal — XLM-R punya byte-fallback).")
    def convert_emoji(text):
        return str(text)

emoji_pattern = r'[\U0001F300-\U0001FAFF\u2600-\u27BF]'
sample_emoji_mask = df[text_col].str.contains(emoji_pattern, regex=True, na=False)
sample_emoji = df[sample_emoji_mask].sample(min(5, sample_emoji_mask.sum()), random_state=2)
display(pd.DataFrame({
    'BEFORE': sample_emoji[text_col],
    'AFTER (emoji)': sample_emoji[text_col].map(convert_emoji)
}))

⚠️ Library 'emoji' belum terinstal. Jalankan: pip install emoji
   Emoji akan DIBIARKAN apa adanya (tidak fatal — XLM-R punya byte-fallback).


,BEFORE,AFTER (emoji)
11943,Wah gila kyknya ktmu orng manipulatif 😭,Wah gila kyknya ktmu orng manipulatif 😭
9351,"SM SukaMauTau DUTA BESAR PALESTINA UNTUK INDONESIA, SEBUT BANTUAN UNTUK PALESTINA DARI RI MASIH TERTAHAN & TUNGGU IZ...","SM SukaMauTau DUTA BESAR PALESTINA UNTUK INDONESIA, SEBUT BANTUAN UNTUK PALESTINA DARI RI MASIH TERTAHAN & TUNGGU IZ..."
8344,🎦 231001 #KUN Weibo Post 🐻🤎 Saya berharap ibu pertiwi damai dan sejahtera ❤❤❤ 🖇*Merayakan HUT ke-74 China #WayV @Way...,🎦 231001 #KUN Weibo Post 🐻🤎 Saya berharap ibu pertiwi damai dan sejahtera ❤❤❤ 🖇*Merayakan HUT ke-74 China #WayV @Way...
17539,"Bagian lengan, paha, pinggang, sama pantat gila sakit sakit semua😩","Bagian lengan, paha, pinggang, sama pantat gila sakit sakit semua😩"
2709,"Majelis Ulama Indonesia (MUI) menerbitkan Fatwa baru, yakni Fatwa Nomor 83 Tahun 2023 tentang Hukum Dukungan terhada...","Majelis Ulama Indonesia (MUI) menerbitkan Fatwa baru, yakni Fatwa Nomor 83 Tahun 2023 tentang Hukum Dukungan terhada..."


## STEP 6 — Normalisasi Slang/Singkatan (Selektif, Berbasis Kamus Tervalidasi dari EDA)
Hanya bentuk yang **benar-benar ditemukan sering muncul** di corpus (hasil EDA sebelumnya) yang dinormalisasi. Kata negasi (`tidak`, `gak`, `nggak`, dst) **TIDAK disentuh** karena maknanya sendiri sudah benar, bukan singkatan yang perlu diubah.

In [9]:
slang_map = {
    'yg':'yang', 'dgn':'dengan', 'dr':'dari', 'krn':'karena', 'karna':'karena',
    'kalo':'kalau', 'kl':'kalau', 'udh':'sudah', 'udah':'sudah', 'blm':'belum',
    'bgt':'banget', 'aja':'saja', 'sm':'sama', 'org':'orang', 'jd':'jadi', 'jgn':'jangan',
    'gw':'saya', 'gue':'saya', 'lu':'kamu', 'lo':'kamu'
}
# Catatan: 'gak','ga','tdk','tak','nggak' SENGAJA TIDAK dimasukkan ke slang_map
# karena merupakan bentuk NEGASI, bukan singkatan biasa -> harus dipertahankan apa adanya.

def normalize_slang(text):
    tokens = str(text).split(' ')
    return ' '.join(slang_map.get(tok.lower(), tok) for tok in tokens)

sample_slang_mask = df[text_col].str.lower().str.contains(r'\b(yg|dgn|krn|kalo|udah|bgt)\b', regex=True, na=False)
sample_slang = df[sample_slang_mask].sample(min(5, sample_slang_mask.sum()), random_state=3)
display(pd.DataFrame({
    'BEFORE': sample_slang[text_col],
    'AFTER (slang)': sample_slang[text_col].map(normalize_slang)
}))

,BEFORE,AFTER (slang)
1881,Zionist itu kayaknya bukan hanya di israel aja tapi ada pada org2 yg berpikiran sama dgn mereka meski berbeda etnis ...,Zionist itu kayaknya bukan hanya di israel saja tapi ada pada org2 yang berpikiran sama dengan mereka meski berbeda ...
17864,"Seorang militer harus taat Sapta Marga prajurit, tapi Prabowo langgar hampir semua pasal yg ada. Prabowo dipecat dgn...","Seorang militer harus taat Sapta Marga prajurit, tapi Prabowo langgar hampir semua pasal yang ada. Prabowo dipecat d..."
20980,"Untuk mengatasi banjir musim hujan dijakarta, wan kibul mantan gubernur DKI, telah menyiapkan TOA? jadi banjir datan...","Untuk mengatasi banjir musim hujan dijakarta, wan kibul mantan gubernur DKI, telah menyiapkan TOA? jadi banjir datan..."
7020,"🚨 PELATIH BRASIL KAGET‼️ ""Ini sangat gila,"" kata Phelipe sambil menggeleng-geleng kepalanya saat melihat kemegahan s...","🚨 PELATIH BRASIL KAGET‼️ ""Ini sangat gila,"" kata Phelipe sambil menggeleng-geleng kepalanya saat melihat kemegahan s..."
7950,yang ku benci adalah sikap Hamas dan Zionisnya. bukan bangsa israel atau bangsa Palestine nya. Dukung sana Dukung si...,yang ku benci adalah sikap Hamas dan Zionisnya. bukan bangsa israel atau bangsa Palestine nya. Dukung sana Dukung si...


## STEP 7 — Gabungkan Semua Langkah Menjadi Satu Pipeline
Semua fungsi step 3–6 digabung jadi satu fungsi `preprocess_text()`, dengan **parameter `lowercase`** yang bisa di-toggle — supaya nanti bisa dibandingkan langsung efeknya terhadap tokenizer XLM-RoBERTa di eksperimen bagian akhir.

In [10]:
def preprocess_text(text, lowercase=False, do_slang=True, do_emoji=True):
    t = clean_url_mention_hashtag(text)
    t = normalize_punctuation(t)
    if do_emoji:
        t = convert_emoji(t)
    if do_slang:
        t = normalize_slang(t)
    if lowercase:
        t = t.lower()
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Dua versi hasil akhir: tetap mempertahankan case asli VS di-lowercase penuh
df['text_clean_cased'] = df[text_col].apply(lambda x: preprocess_text(x, lowercase=False))
df['text_clean_lower'] = df[text_col].apply(lambda x: preprocess_text(x, lowercase=True))

print("Contoh hasil pipeline lengkap (BEFORE vs 2 versi AFTER):")
display(df[[text_col, 'text_clean_cased', 'text_clean_lower']].sample(5, random_state=7))

Contoh hasil pipeline lengkap (BEFORE vs 2 versi AFTER):


,text,text_clean_cased,text_clean_lower
18030,"Israel Tolak Resolusi Gencatan Senjata, Netanyahu: Tidak Akan Pernah Terjadi","Israel Tolak Resolusi Gencatan Senjata, Netanyahu: Tidak Akan Pernah Terjadi","israel tolak resolusi gencatan senjata, netanyahu: tidak akan pernah terjadi"
4702,Kenapa di fyp gw lewat couple gay semua,Kenapa di fyp saya lewat couple gay semua,kenapa di fyp saya lewat couple gay semua
5117,“Soimah Pancawati berbagi hadiah melalui akun Facebook Mae Soimah Pancawati”,“Soimah Pancawati berbagi hadiah melalui akun Facebook Mae Soimah Pancawati”,“soimah pancawati berbagi hadiah melalui akun facebook mae soimah pancawati”
7441,"Hasil Pengeluaran Pasaran CHINA Hari Kamis, 09 November 2023 Prize 1 : 2671 Prize 2 : 2455 Prize 3 : 7924 Shio : ULA...","Hasil Pengeluaran Pasaran CHINA Hari Kamis, 09 November 2023 Prize 1 : 2671 Prize 2 : 2455 Prize 3 : 7924 Shio : ULA...","hasil pengeluaran pasaran china hari kamis, 09 november 2023 prize 1 : 2671 prize 2 : 2455 prize 3 : 7924 shio : ula..."
4926,Jasa skripsi tesis disertasi jurusan joki Informatika Sistem informa hukum Jurusan ilkom hubungan internasional Pert...,Jasa skripsi tesis disertasi jurusan joki Informatika Sistem informa hukum Jurusan ilkom hubungan internasional Pert...,jasa skripsi tesis disertasi jurusan joki informatika sistem informa hukum jurusan ilkom hubungan internasional pert...


## STEP 8 — Train-Test Split (Setelah Dedup, Stratified pada Label Utama)
Split dilakukan **setelah** deduplikasi (Step 2) agar tidak ada salinan/near-duplicate yang bocor antara train dan test.

In [11]:
from sklearn.model_selection import train_test_split

df_model = df.dropna(subset=[main_label_col]).copy()  # buang baris tanpa label valid (jika ada)

X_train, X_test, y_train, y_test = train_test_split(
    df_model[['text_id', text_col, 'text_clean_cased', 'text_clean_lower']],
    df_model[main_label_col],
    test_size=0.2,
    random_state=42,
    stratify=df_model[main_label_col]
)

print(f"Train: {len(X_train):,} baris | Test: {len(X_test):,} baris")
print("\nDistribusi label di TRAIN:")
print(y_train.value_counts(normalize=True).round(3))
print("\nDistribusi label di TEST:")
print(y_test.value_counts(normalize=True).round(3))

Train: 20,798 baris | Test: 5,200 baris

Distribusi label di TRAIN:
toxicity_label
0.0    0.866
1.0    0.070
0.5    0.064
Name: proportion, dtype: float64

Distribusi label di TEST:
toxicity_label
0.0    0.867
1.0    0.070
0.5    0.064
Name: proportion, dtype: float64


---
## 🔬 EKSPERIMEN — Pengaruh Lowercasing terhadap Tokenisasi XLM-RoBERTa

### Latar Belakang Teknis
XLM-RoBERTa **tidak seperti BERT-base-uncased**. Perbedaan kuncinya:

| | BERT-base-**uncased** | **XLM-RoBERTa** |
|---|---|---|
| Tokenizer | WordPiece, dilatih dari korpus yang **sudah di-lowercase** | **SentencePiece (Unigram)**, dilatih dari korpus **mixed-case asli** (CC-100, 100 bahasa) |
| Vocabulary | Hanya berisi bentuk huruf kecil | Berisi bentuk huruf besar **dan** kecil sebagai token terpisah |
| Karakter di luar vocab | Bisa menghasilkan `[UNK]` | Ada **byte-fallback**, nyaris tidak pernah `<unk>` |
| Wajib lowercase manual? | **Ya**, karena model memang dilatih begitu | **Tidak** — model sudah "belajar" kedua case sekaligus |

**Hipotesis:** Karena XLM-R sudah dilatih dari teks apa adanya (termasuk huruf besar), melakukan lowercasing manual **tidak memberi keuntungan berarti** dari sisi tokenisasi, dan berisiko **menghilangkan sinyal ALLCAPS/penekanan emosi** yang relevan untuk deteksi toxicity (temuan EDA: ALLCAPS muncul di 27,9%–37,3% dokumen tergantung kelas).

Sel-sel di bawah menguji hipotesis ini secara **empiris** pada data kita sendiri.

In [12]:
# === Percobaan 1: Tokenisasi asli dengan XLM-RoBERTa tokenizer ===
# Membutuhkan koneksi internet (untuk mengunduh tokenizer) & library transformers+sentencepiece:
#   pip install transformers sentencepiece

TOKENIZER_READY = False
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
    TOKENIZER_READY = True
    print("✅ Tokenizer 'xlm-roberta-base' berhasil dimuat.")
except Exception as e:
    print("⚠️ Tokenizer XLM-R tidak dapat dimuat di lingkungan ini.")
    print("   Alasan:", e)
    print("   -> Jalankan: pip install transformers sentencepiece (butuh koneksi internet)")
    print("   -> Sel berikutnya tetap bisa dijalankan sebagai analisis PROXY tanpa transformers.")

✅ Tokenizer 'xlm-roberta-base' berhasil dimuat.


In [13]:
if TOKENIZER_READY:
    sample_n = min(2000, len(df))
    sample_df = df.sample(sample_n, random_state=42)

    def token_stats(texts):
        lengths, unk_counts = [], []
        for t in texts:
            ids = tokenizer.encode(t, add_special_tokens=False)
            lengths.append(len(ids))
            unk_counts.append(sum(1 for i in ids if i == tokenizer.unk_token_id))
        return np.array(lengths), np.array(unk_counts)

    len_cased, unk_cased = token_stats(sample_df['text_clean_cased'])
    len_lower, unk_lower = token_stats(sample_df['text_clean_lower'])

    result_table = pd.DataFrame({
        'Metrik': ['Rata-rata jumlah subword token', 'Median jumlah subword token',
                   'Total token <unk>', '% dokumen dgn token <unk>'],
        'CASED (tanpa lowercase)': [
            round(len_cased.mean(), 2), int(np.median(len_cased)),
            int(unk_cased.sum()), round((unk_cased > 0).mean() * 100, 3)
        ],
        'LOWERCASE': [
            round(len_lower.mean(), 2), int(np.median(len_lower)),
            int(unk_lower.sum()), round((unk_lower > 0).mean() * 100, 3)
        ]
    })
    display(result_table)

    pct_diff = (len_lower.mean() - len_cased.mean()) / len_cased.mean() * 100
    print(f"\nSelisih rata-rata panjang token (lower vs cased): {pct_diff:+.2f}%")

    # Contoh konkret perbedaan subword pada kata ALLCAPS
    print("\nContoh tokenisasi kata ALLCAPS vs lowercase:")
    for w in ['GILA', 'gila', 'ANJING', 'anjing', 'BODOH', 'bodoh']:
        pieces = tokenizer.tokenize(w)
        print(f"  '{w}' -> {pieces}  ({len(pieces)} subword)")
else:
    print("Lewati sel ini — TOKENIZER_READY = False. Lihat analisis proxy di sel berikutnya.")

Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors


,Metrik,CASED (tanpa lowercase),LOWERCASE
0,Rata-rata jumlah subword token,74.28,72.75
1,Median jumlah subword token,36.00,36.00
2,Total token <unk>,148.00,148.00
3,% dokumen dgn token <unk>,4.40,4.40



Selisih rata-rata panjang token (lower vs cased): -2.07%

Contoh tokenisasi kata ALLCAPS vs lowercase:
  'GILA' -> ['▁G', 'ILA']  (2 subword)
  'gila' -> ['▁gila']  (1 subword)
  'ANJING' -> ['▁A', 'NJI', 'NG']  (3 subword)
  'anjing' -> ['▁anjing']  (1 subword)
  'BODOH' -> ['▁BO', 'DO', 'H']  (3 subword)
  'bodoh' -> ['▁bodoh']  (1 subword)


**Cara membaca hasil di atas:**
- Jika rata-rata jumlah token **LOWERCASE lebih sedikit atau setara** dengan **CASED**, berarti lowercasing tidak memberi penghematan berarti pada panjang sequence.
- Jika contoh tokenisasi kata ALLCAPS (`GILA`, `ANJING`, dll) pecah menjadi **lebih banyak subword** dibanding versi lowercase-nya, itu bukti bahwa vocab XLM-R memang lebih "akrab" dengan bentuk lowercase (karena korpus pretraining mayoritas huruf kecil) — TAPI ini **bukan berarti harus di-lowercase**, karena informasi "kata ini ditulis ALLCAPS" itu sendiri adalah sinyal (emosi/penekanan) yang hilang kalau kita paksa lowercase semua.

In [14]:
# === Percobaan 2 (PROXY, tidak butuh transformers/internet) ===
# Analisis vocabulary sederhana berbasis regex sebagai gambaran kasar
# tentang efek lowercasing terhadap ukuran vocabulary di LEVEL KATA (bukan subword asli XLM-R).

def word_vocab(texts):
    tokens = re.findall(r'\b\w+\b', ' '.join(texts.astype(str)))
    return Counter(tokens)

vocab_cased = word_vocab(df['text_clean_cased'])
vocab_lower = word_vocab(df['text_clean_lower'])

proxy_table = pd.DataFrame({
    'Metrik': ['Ukuran vocabulary (kata unik)', 'Total token'],
    'CASED (tanpa lowercase)': [len(vocab_cased), sum(vocab_cased.values())],
    'LOWERCASE': [len(vocab_lower), sum(vocab_lower.values())]
})
display(proxy_table)

reduction_pct = (len(vocab_cased) - len(vocab_lower)) / len(vocab_cased) * 100
print(f"\nLowercasing mengurangi ukuran vocabulary kata sebesar {reduction_pct:.2f}% "
      f"(karena 'GILA' dan 'gila' digabung jadi satu entri).")

# Contoh kata yang 'hilang identitasnya' akibat lowercasing
allcaps_words = {w for w in vocab_cased if w.isupper() and len(w) >= 3}
print(f"\nJumlah kata ALLCAPS unik (≥3 huruf) yang identitas huruf-besarnya akan hilang jika di-lowercase: {len(allcaps_words):,}")
print("Contoh:", list(allcaps_words)[:15])

,Metrik,CASED (tanpa lowercase),LOWERCASE
0,Ukuran vocabulary (kata unik),109719,84795
1,Total token,1159673,1159691



Lowercasing mengurangi ukuran vocabulary kata sebesar 22.72% (karena 'GILA' dan 'gila' digabung jadi satu entri).

Jumlah kata ALLCAPS unik (≥3 huruf) yang identitas huruf-besarnya akan hilang jika di-lowercase: 15,093
Contoh: ['PRCAJO', 'HALF', 'KENDARNFO', 'CRIME', 'MAZE', 'PUJIASTUTI', 'AKHLAK', 'DUDA', 'BANDUNGTOTO', 'TAKTIK', 'SAMPE', 'MAUMIN', '25JUTAAN', 'HANGZHOU', 'DEAL']


**💡 Insight & Kesimpulan Eksperimen:**

1. **Dari sisi teknis tokenizer**: XLM-RoBERTa memakai SentencePiece Unigram dengan byte-fallback yang dilatih dari korpus *mixed-case* asli — model ini **tidak wajib** menerima input lowercase seperti BERT-uncased. Tokenisasi tetap berjalan normal (hampir tidak ada `<unk>`) baik teks di-lowercase maupun tidak.
2. **Dari sisi ukuran vocabulary kata**: lowercasing memang mengurangi jumlah kata unik (menggabungkan `GILA`/`gila` jadi satu bentuk), tapi penurunan ini **tidak signifikan menyelesaikan masalah sparsity** karena XLM-R sudah bekerja di level subword, bukan whole-word seperti BoW/TF-IDF.
3. **Trade-off yang lebih penting**: dataset ini punya **27,9%–37,3% dokumen dengan ALLCAPS**, dan dari EDA sebelumnya elemen ekspresif seperti ini berkorelasi dengan pola emosi pada teks toxic. Lowercasing **menghilangkan sinyal ini secara permanen** sebelum model sempat mempelajarinya.
4. **Rekomendasi akhir:** Untuk fine-tuning **XLM-RoBERTa** pada tugas hate speech ini, **JANGAN melakukan lowercasing** pada teks input. Pertahankan case asli (`text_clean_cased`), karena:
   - Tidak ada keuntungan teknis berarti dari sisi tokenizer (beda dengan BERT-uncased).
   - Case asli (termasuk ALLCAPS) berpotensi menjadi sinyal fitur yang berguna untuk deteksi intensitas/emosi dalam ujaran kebencian.
   - Jika ingin tetap menangkap efek "penekanan" tanpa kehilangan normalisasi kata, alternatif yang lebih baik adalah **menambahkan fitur terpisah** (mis. rasio huruf kapital per dokumen) dibanding memaksa lowercase seluruh teks.

## Bagian Akhir — Ekspor Data Hasil Preprocessing

In [15]:
import os

os.makedirs('data/interim', exist_ok=True)

train_export = X_train.copy(); train_export[main_label_col] = y_train
test_export = X_test.copy(); test_export[main_label_col] = y_test

train_export.to_csv('data/interim/train_preprocessed.csv', index=False)
test_export.to_csv('data/interim/test_preprocessed.csv', index=False)

print(f"✅ Data preprocessing berhasil diekspor:")
print(f"   data/interim/train_preprocessed.csv  ({len(train_export):,} baris)")
print(f"   data/interim/test_preprocessed.csv   ({len(test_export):,} baris)")
print(f"\nKolom teks yang direkomendasikan untuk fine-tuning XLM-RoBERTa: 'text_clean_cased' (TANPA lowercasing).")

✅ Data preprocessing berhasil diekspor:
   data/interim/train_preprocessed.csv  (20,798 baris)
   data/interim/test_preprocessed.csv   (5,200 baris)

Kolom teks yang direkomendasikan untuk fine-tuning XLM-RoBERTa: 'text_clean_cased' (TANPA lowercasing).


## 📌 Ringkasan Pipeline Preprocessing

| Langkah | Status | Catatan |
|---|---|---|
| Buang teks tidak valid | ✅ Dieksekusi | Dampak minimal (~1 baris) |
| Deduplikasi + resolusi conflicting label | ✅ Dieksekusi | Sebelum split, cegah data leakage |
| URL → `<URL>`, Mention → `<USER>` | ✅ Dieksekusi | Kurangi noise, pertahankan sinyal keberadaan tautan/sebutan |
| Hashtag: buang simbol `#`, simpan kata | ✅ Dieksekusi | Topik tetap terbaca model |
| Newline & whitespace | ✅ Dieksekusi | Normalisasi struktural, tidak mengubah makna |
| Tanda baca berlebihan | ✅ Dieksekusi (dibatasi maks 2x) | Sinyal intensitas tetap ada tapi tidak liar |
| Emoji | ✅ Dikonversi ke token (jika library tersedia) | Sinyal emosi dipertahankan |
| Slang/singkatan | ✅ Dieksekusi selektif | Negasi TIDAK disentuh |
| Stopword removal | ❌ Tidak dilakukan | XLM-R tidak butuh; hanya relevan untuk model klasik |
| Stemming | ❌ Tidak dilakukan | Berisiko merusak makna istilah spesifik |
| **Lowercasing** | ❌ **Tidak dilakukan** | **Terbukti dari eksperimen: tidak menguntungkan untuk XLM-R, malah berisiko hilangkan sinyal ALLCAPS** |
| Train-test split | ✅ Stratified, setelah dedup | Distribusi kelas terjaga di train & test |
